In [14]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

In [5]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TW_1500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1500/Twitter-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns
TW_1000.columns = columns
TW_1500.columns = columns


In [6]:
top_features = ['NCD','NAD','NAC','AS(NAC)','NA', 'label']

selected_columns = [
    col for col in TW_500.columns 
    if any(col.startswith(f + '_') for f in top_features)
]

# create new dataset
Selected_500 = TW_500[selected_columns]
Selected_500['label']=TW_500['label']

print("Number of features:", len(selected_columns))



top_features = ['NCD','NAC', 'AS(NAC)', 'AS(NA)', 'NAD','label']

selected_columns = [
    col for col in TW_1000.columns 
    if any(col.startswith(f + '_') for f in top_features)
]

Selected_1000 = TW_1000[selected_columns]
Selected_1000['label']=TW_1000['label']

print("Number of features:", len(selected_columns))


top_features = ['NCD','NAC', 'NAD', 'AS(NAC)', 'AS(NA)','label']

selected_columns = [
    col for col in TW_1500.columns 
    if any(col.startswith(f + '_') for f in top_features)
]

Selected_1500 = TW_1500[selected_columns]
Selected_1500['label']=TW_1500['label']
print("Number of features:", len(selected_columns))

Number of features: 35
Number of features: 35
Number of features: 35


### priprava

In [7]:
X = Selected_500.drop(columns=['label'])
y = Selected_500['label']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y,random_state=42)

### 1.Random Forest with Grid Search

In [8]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print("Random Forest with Grid Search")
print(best_rf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest with Grid Search
RandomForestClassifier(min_samples_split=5, n_estimators=200, random_state=42)
Accuracy: 0.9836187904200128
Precision: 0.7981859410430839
Recall: 0.4861878453038674
F1: 0.6042918454935622
ROC-AUC: 0.9660254142034369


### XGBoost with Grid Search

In [9]:
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb,param_grid_xgb, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost with Grid Search")
print(best_xgb)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost with Grid Search
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)
Accuracy: 0.9823395636415322
Precision: 0.7379454926624738
Recall: 0.4861878453038674
F1: 0.5861781848459617
ROC-AUC: 0.9702248522868189


### XGBoost randomizer

In [10]:

xgb = XGBClassifier(random_state=42,eval_metric='logloss',use_label_encoder=False)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

random_search = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=30,  scoring='f1', cv=5,verbose=1,random_state=42,n_jobs=-1).fit(X_train, y_train)

best_xgb_rand = random_search.best_estimator_

print("Best parameters:", random_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'subsample': 0.8, 'reg_lambda': 2, 'reg_alpha': 1, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 1.0}
Accuracy: 0.9823395636415322
Precision: 0.7379454926624738
Recall: 0.4861878453038674
F1: 0.5861781848459617
ROC-AUC: 0.9702248522868189


### Logistic Regression with Scaling and Grid Search

In [11]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(solver="liblinear", max_iter=1000))
])

param_dist = {
    "lr__C": [0.1, 1, 10],
    "lr__penalty": ["l1", "l2"]
}

search = RandomizedSearchCV( pipe_lr,param_distributions=param_dist,n_iter=4,cv=3,scoring="f1",n_jobs=-1,random_state=42,verbose=2)

search.fit(X_train, y_train)

best_lr = search.best_estimator_

y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:, 1]

print(best_lr)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(C=10, max_iter=1000, solver='liblinear'))])
Accuracy: 0.9805983938597115
Precision: 0.7354497354497355
Recall: 0.3839779005524862
F1: 0.5045372050816697
ROC-AUC: 0.9513108701022717


### SVM

In [15]:

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(max_iter=2000))
])

param_dist = {
    "svm__C": [0.1, 1, 10]
}

search = RandomizedSearchCV(
    pipe_svm,
    param_distributions=param_dist,
    n_iter=3,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=42,
    verbose=2
)

search.fit(X_train, y_train)

best_svm = search.best_estimator_

y_pred = best_svm.predict(X_test)

print(best_svm)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Pipeline(steps=[('scaler', StandardScaler()),
                ('svm', LinearSVC(C=10, max_iter=2000))])
Accuracy: 0.9804562575509914
Precision: 0.8
Recall: 0.32044198895027626
F1: 0.45759368836291914


### STACKING MODEL Random Forest, XGBoost, Logisticka Regresia

In [16]:

estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9838319948830929
Precision: 0.7930283224400871
Recall: 0.5027624309392266
F1: 0.6153846153846154
ROC-AUC: 0.9710420302990858


### STACKING MODEL with Random Forest, Random XGBoost, Logisticka Regresia

In [18]:
estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb_rand ),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9837253926515528
Precision: 0.7854077253218884
Recall: 0.505524861878453
F1: 0.6151260504201681
ROC-AUC: 0.9707614346989053


### STACKING MODEL with Random Forest, SVM, Logisticka Regresia

In [21]:
estimators = [
    ('rf', best_rf),
    ('svm', best_svm),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier( estimators=estimators,final_estimator=meta_model, cv=5,n_jobs=-1, stack_method='auto'  )
stack_model.fit(X_train, y_train)


y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]


print("STACKING (RF + SVM + LR)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING (RF + SVM + LR)
Accuracy: 0.9836898585743729
Precision: 0.7924944812362031
Recall: 0.4958563535911602
F1: 0.6100254885301615
ROC-AUC: 0.958666101915546


### 1000

In [23]:
X = Selected_1000.drop(columns=['label'])
y = Selected_1000['label']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y,random_state=42)

### 1.Random Forest with Grid Search

In [24]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print("Random Forest with Grid Search")
print(best_rf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest with Grid Search
RandomForestClassifier(min_samples_split=5, n_estimators=200, random_state=42)
Accuracy: 0.995487172198138
Precision: 0.8333333333333334
Recall: 0.574468085106383
F1: 0.6801007556675063
ROC-AUC: 0.9683017835073789


### XGBoost with Grid Search

In [25]:
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb,param_grid_xgb, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost with Grid Search")
print(best_xgb)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost with Grid Search
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)
Accuracy: 0.9948830928860778
Precision: 0.7407407407407407
Recall: 0.5957446808510638
F1: 0.660377358490566
ROC-AUC: 0.9826777998961597


### XGBoost randomizer

In [26]:
xgb = XGBClassifier(random_state=42,eval_metric='logloss',use_label_encoder=False)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

random_search = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=30,  scoring='f1', cv=5,verbose=1,random_state=42,n_jobs=-1).fit(X_train, y_train)

best_xgb_rand = random_search.best_estimator_

print("Best parameters:", random_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'subsample': 0.7, 'reg_lambda': 1.5, 'reg_alpha': 1, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 10, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.9}
Accuracy: 0.9948830928860778
Precision: 0.7407407407407407
Recall: 0.5957446808510638
F1: 0.660377358490566
ROC-AUC: 0.9826777998961597


### Logistic Regression with Scaling and Grid Search

In [27]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(solver="liblinear", max_iter=1000))
])

param_dist = {
    "lr__C": [0.1, 1, 10],
    "lr__penalty": ["l1", "l2"]
}

search = RandomizedSearchCV( pipe_lr,param_distributions=param_dist,n_iter=4,cv=3,scoring="f1",n_jobs=-1,random_state=42,verbose=2)

search.fit(X_train, y_train)

best_lr = search.best_estimator_

y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:, 1]

print(best_lr)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(C=10, max_iter=1000, solver='liblinear'))])
Accuracy: 0.9939592068793973
Precision: 0.7338129496402878
Recall: 0.4340425531914894
F1: 0.5454545454545454
ROC-AUC: 0.9593891260409765


### SVM

In [28]:

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(max_iter=2000))
])

param_dist = {
    "svm__C": [0.1, 1, 10]
}

search = RandomizedSearchCV(
    pipe_svm,
    param_distributions=param_dist,
    n_iter=3,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=42,
    verbose=2
)

search.fit(X_train, y_train)

best_svm = search.best_estimator_

y_pred = best_svm.predict(X_test)

print(best_svm)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Pipeline(steps=[('scaler', StandardScaler()),
                ('svm', LinearSVC(C=1, max_iter=2000))])
Accuracy: 0.9941368772652974
Precision: 0.8571428571428571
Recall: 0.3574468085106383
F1: 0.5045045045045045


### STACKING MODEL Random Forest, XGBoost, Logisticka Regresia

In [29]:

estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9955582403524981
Precision: 0.8089887640449438
Recall: 0.6127659574468085
F1: 0.6973365617433414
ROC-AUC: 0.9830548119933304


### STACKING MODEL with Random Forest, Random XGBoost, Logisticka Regresia

In [30]:
estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb_rand ),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9955937744296781
Precision: 0.8135593220338984
Recall: 0.6127659574468085
F1: 0.6990291262135923
ROC-AUC: 0.9836555916345247


### STACKING MODEL with Random Forest, SVM, Logisticka Regresia

In [31]:
estimators = [
    ('rf', best_rf),
    ('svm', best_svm),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier( estimators=estimators,final_estimator=meta_model, cv=5,n_jobs=-1, stack_method='auto'  )
stack_model.fit(X_train, y_train)


y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]


print("STACKING (RF + SVM + LR)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING (RF + SVM + LR)
Accuracy: 0.995629308506858
Precision: 0.8111111111111111
Recall: 0.6212765957446809
F1: 0.7036144578313253
ROC-AUC: 0.9621877832832302


### 1500

In [32]:
X = Selected_1500.drop(columns=['label'])
y = Selected_1500['label']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y,random_state=42)

### 1.Random Forest with Grid Search

In [34]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print("Random Forest with Grid Search")
print(best_rf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest with Grid Search
RandomForestClassifier(min_samples_split=5, n_estimators=200, random_state=42)
Accuracy: 0.997690284983299
Precision: 0.8
Recall: 0.4489795918367347
F1: 0.5751633986928104
ROC-AUC: 0.97220475695627


### XGBoost with Grid Search

In [36]:
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb,param_grid_xgb, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost with Grid Search")
print(best_xgb)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost with Grid Search
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)
Accuracy: 0.9975126145973989
Precision: 0.7333333333333333
Recall: 0.4489795918367347
F1: 0.5569620253164557
ROC-AUC: 0.9925061273974717


### XGBoost randomizer

In [38]:

xgb = XGBClassifier(random_state=42,eval_metric='logloss',use_label_encoder=False)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

random_search = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=30,  scoring='f1', cv=5,verbose=1,random_state=42,n_jobs=-1).fit(X_train, y_train)

best_xgb_rand = random_search.best_estimator_

print("Best parameters:", random_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}
Accuracy: 0.9975126145973989
Precision: 0.7333333333333333
Recall: 0.4489795918367347
F1: 0.5569620253164557
ROC-AUC: 0.9925061273974717


### Logistic Regression with Scaling and Grid Search

In [40]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(solver="liblinear", max_iter=1000))
])

param_dist = {
    "lr__C": [0.1, 1, 10],
    "lr__penalty": ["l1", "l2"]
}

search = RandomizedSearchCV( pipe_lr,param_distributions=param_dist,n_iter=4,cv=3,scoring="f1",n_jobs=-1,random_state=42,verbose=2)

search.fit(X_train, y_train)

best_lr = search.best_estimator_

y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:, 1]

print(best_lr)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(C=1, max_iter=1000, penalty='l1',
                                    solver='liblinear'))])
Accuracy: 0.997548148674579
Precision: 0.7101449275362319
Recall: 0.5
F1: 0.5868263473053892
ROC-AUC: 0.9820362462486064


### SVM

In [42]:

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(max_iter=2000))
])

param_dist = {
    "svm__C": [0.1, 1, 10]
}

search = RandomizedSearchCV(
    pipe_svm,
    param_distributions=param_dist,
    n_iter=3,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=42,
    verbose=2
)

search.fit(X_train, y_train)

best_svm = search.best_estimator_

y_pred = best_svm.predict(X_test)

print(best_svm)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Pipeline(steps=[('scaler', StandardScaler()),
                ('svm', LinearSVC(C=10, max_iter=2000))])
Accuracy: 0.9974060123658589
Precision: 0.7358490566037735
Recall: 0.3979591836734694
F1: 0.5165562913907285


### STACKING MODEL Random Forest, XGBoost, Logisticka Regresia

In [44]:

estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9980456257550991
Precision: 0.7945205479452054
Recall: 0.5918367346938775
F1: 0.6783625730994152
ROC-AUC: 0.9952025097587175


### STACKING MODEL with Random Forest, Random XGBoost, Logisticka Regresia

In [46]:
estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb_rand ),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.997974557600739
Precision: 0.7808219178082192
Recall: 0.5816326530612245
F1: 0.6666666666666666
ROC-AUC: 0.994601413522191


### STACKING MODEL with Random Forest, SVM, Logisticka Regresia

In [48]:
estimators = [
    ('rf', best_rf),
    ('svm', best_svm),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier( estimators=estimators,final_estimator=meta_model, cv=5,n_jobs=-1, stack_method='auto'  )
stack_model.fit(X_train, y_train)


y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]


print("STACKING (RF + SVM + LR)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING (RF + SVM + LR)
Accuracy: 0.997654750906119
Precision: 0.7666666666666667
Recall: 0.46938775510204084
F1: 0.5822784810126582
ROC-AUC: 0.9770790215958012
